In [ ]:
!pip install torch numpy matplotlib tqdm pretty_midi
!pip install torch numpy matplotlib tqdm mido

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 36.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 2.9 MB/s eta 0:00:00
  Created wheel for pretty_midi: filename=pretty_midi-0.2.11-py3-none-any.whl size=5595886 sha256=bbb520c3fae252304fa529c830c3092f72f37ea29ed28c38f7efbad688cd2f91
  Stored in directory: /root/.cache/pip/wheels/f4/ad/93/a7042fe12668827574927ade9deec7f29aad2a1001b1501882
Successfully built pretty_midi


In [ ]:
!unzip -q "MPD-Set_ori.zip" -d "/content"
!ls -la /content/MPD-Set_ori | head

total 132
drwxr-xr-x 6 root root  4096 Jan 29 08:29 .
drwxr-xr-x 1 root root  4096 Jan 29 19:10 ..
-rw-r--r-- 1 root root  8196 Jan 29 08:29 .DS_Store
drwxr-xr-x 2 root root 24576 Jan 29 08:27 duration
drwxr-xr-x 2 root root 24576 Jan 29 08:27 melody
drwxr-xr-x 2 root root 24576 Jan 29 08:27 pitch
drwxr-xr-x 2 root root 24576 Jan 29 08:27 shuffle


In [ ]:
!unzip -q "MPD-Set_plag.zip" -d "/content"
!ls -la /content/MPD-Set_plag.zip | head

-rw-r--r-- 1 root root 1203779 Jan 29 19:07 /content/MPD-Set_plag.zip


In [ ]:
"""
LSTM + Triplet Loss для виявлення музичного плагіату (MIDI)
===========================================================

Оновлена версія для структури MPD-Set з підпапками:

MPD-Set_ori/
├── duration/
├── melody/
├── pitch/
└── shuffle/

MPD-Set_plag/
├── duration/
├── melody/
├── pitch/
└── shuffle/

Встановлення:
    pip install torch numpy matplotlib tqdm pretty_midi

Запуск:
    python midi_plagiarism_lstm_triplet.py
"""

import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from typing import List, Tuple, Dict, Optional
import random
from collections import defaultdict
import matplotlib.pyplot as plt
from tqdm import tqdm

# MIDI парсери
try:
    import pretty_midi
    PRETTY_MIDI_AVAILABLE = True
except ImportError:
    PRETTY_MIDI_AVAILABLE = False
    print("⚠️ pretty_midi не встановлено. Встановіть: pip install pretty_midi")

try:
    import mido
    MIDO_AVAILABLE = True
except ImportError:
    MIDO_AVAILABLE = False


# ==============================================================================
# 1. КОНФІГУРАЦІЯ
# ==============================================================================

class Config:
    """Конфігурація"""

    # Шляхи - ОНОВЛЕНО для твоєї структури
    DATA_DIR = "./data/midi"
    ORI_DIR = os.path.join(DATA_DIR, "MPD-Set_ori")
    PLAG_DIR = os.path.join(DATA_DIR, "MPD-Set_plag")

    # Типи плагіату (підпапки)
    PLAGIARISM_TYPES = ["duration", "melody", "pitch", "shuffle"]

    # MIDI параметри
    SAMPLE_RATE = 100       # Герц для piano roll
    MAX_SEQ_LENGTH = 500    # Максимальна довжина

    # Модель
    INPUT_SIZE = 128        # 128 MIDI нот
    HIDDEN_SIZE = 64
    NUM_LAYERS = 2
    EMBEDDING_SIZE = 64
    DROPOUT = 0.3
    BIDIRECTIONAL = True

    # Тренування
    BATCH_SIZE = 16
    LEARNING_RATE = 0.001
    NUM_EPOCHS = 20
    MARGIN = 1.0
    PATIENCE = 10

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    MODEL_SAVE_PATH = "./models"


# ==============================================================================
# 2. MIDI ПАРСЕР
# ==============================================================================

class MidiParser:
    """Парсер MIDI файлів"""

    def __init__(self, max_length: int = Config.MAX_SEQ_LENGTH,
                 sample_rate: int = Config.SAMPLE_RATE):
        self.max_length = max_length
        self.sample_rate = sample_rate

    def parse_midi(self, filepath: str) -> Optional[np.ndarray]:
        """Парсить MIDI файл у piano roll"""

        if PRETTY_MIDI_AVAILABLE:
            try:
                midi = pretty_midi.PrettyMIDI(filepath)
                piano_roll = midi.get_piano_roll(fs=self.sample_rate)
                piano_roll = piano_roll.T  # [time, 128]
                piano_roll = piano_roll / 127.0  # Нормалізація
                return self._pad_or_truncate(piano_roll).astype(np.float32)
            except Exception as e:
                print(f"  ⚠️ Помилка pretty_midi для {os.path.basename(filepath)}: {e}")

        if MIDO_AVAILABLE:
            try:
                return self._parse_with_mido(filepath)
            except Exception as e:
                print(f"  ⚠️ Помилка mido для {os.path.basename(filepath)}: {e}")

        return None

    def _parse_with_mido(self, filepath: str) -> Optional[np.ndarray]:
        """Альтернативний парсер через mido"""
        mid = mido.MidiFile(filepath)

        # Обчислюємо загальну тривалість
        total_time = 0
        for track in mid.tracks:
            track_time = 0
            for msg in track:
                track_time += msg.time
            total_time = max(total_time, track_time)

        # Конвертуємо в секунди (приблизно)
        ticks_per_beat = mid.ticks_per_beat
        tempo = 500000  # default tempo

        for track in mid.tracks:
            for msg in track:
                if msg.type == 'set_tempo':
                    tempo = msg.tempo
                    break

        seconds_per_tick = tempo / (ticks_per_beat * 1000000)
        duration_sec = total_time * seconds_per_tick

        # Створюємо piano roll
        num_frames = int(duration_sec * self.sample_rate) + 1
        num_frames = min(num_frames, self.max_length * 2)  # Обмежуємо
        piano_roll = np.zeros((num_frames, 128))

        for track in mid.tracks:
            current_tick = 0
            for msg in track:
                current_tick += msg.time
                current_time = current_tick * seconds_per_tick
                frame = int(current_time * self.sample_rate)

                if frame < num_frames:
                    if msg.type == 'note_on' and msg.velocity > 0:
                        piano_roll[frame, msg.note] = msg.velocity / 127.0

        return self._pad_or_truncate(piano_roll).astype(np.float32)

    def _pad_or_truncate(self, data: np.ndarray) -> np.ndarray:
        """Padding або truncate до max_length"""
        if len(data) > self.max_length:
            return data[:self.max_length]
        elif len(data) < self.max_length:
            padding = np.zeros((self.max_length - len(data), data.shape[1]))
            return np.vstack([data, padding])
        return data


# ==============================================================================
# 3. ЗАВАНТАЖЕННЯ ДАТАСЕТУ З ПІДПАПКАМИ
# ==============================================================================

def extract_case_number(filename: str) -> str:
    """Витягує номер кейсу з назви файлу"""
    basename = os.path.splitext(filename)[0]
    # Case0001_xxx.mid -> Case0001
    if "Case" in basename:
        parts = basename.split("_")
        for part in parts:
            if part.startswith("Case"):
                return part
    return basename


def load_midi_dataset_with_subfolders(
    ori_dir: str,
    plag_dir: str,
    parser: MidiParser,
    plagiarism_types: List[str] = Config.PLAGIARISM_TYPES
) -> Dict[str, Dict[str, np.ndarray]]:
    """
    Завантажує MIDI датасет зі структурою підпапок

    Структура:
    ori_dir/
    ├── duration/
    ├── melody/
    ├── pitch/
    └── shuffle/

    plag_dir/
    ├── duration/
    ├── melody/
    ├── pitch/
    └── shuffle/
    """

    dataset = defaultdict(lambda: {"original": None, "plagiarism": None, "type": None})

    for plag_type in plagiarism_types:
        ori_subdir = os.path.join(ori_dir, plag_type)
        plag_subdir = os.path.join(plag_dir, plag_type)

        if not os.path.exists(ori_subdir):
            print(f"⚠️ Не знайдено: {ori_subdir}")
            continue
        if not os.path.exists(plag_subdir):
            print(f"⚠️ Не знайдено: {plag_subdir}")
            continue

        print(f"\n📁 Завантаження типу: {plag_type}")

        # Отримуємо списки файлів
        ori_files = {extract_case_number(f): f
                     for f in os.listdir(ori_subdir)
                     if f.lower().endswith(('.mid', '.midi'))}

        plag_files = {extract_case_number(f): f
                      for f in os.listdir(plag_subdir)
                      if f.lower().endswith(('.mid', '.midi'))}

        # Знаходимо спільні кейси
        common_cases = set(ori_files.keys()) & set(plag_files.keys())
        print(f"   Оригіналів: {len(ori_files)}, Плагіатів: {len(plag_files)}, Пар: {len(common_cases)}")

        # Завантажуємо пари
        for case_num in tqdm(common_cases, desc=f"   {plag_type}", leave=False):
            # Унікальний ключ: тип + номер кейсу
            key = f"{plag_type}_{case_num}"

            # Оригінал
            ori_path = os.path.join(ori_subdir, ori_files[case_num])
            ori_features = parser.parse_midi(ori_path)

            # Плагіат
            plag_path = os.path.join(plag_subdir, plag_files[case_num])
            plag_features = parser.parse_midi(plag_path)

            if ori_features is not None and plag_features is not None:
                dataset[key] = {
                    "original": ori_features,
                    "plagiarism": plag_features,
                    "type": plag_type
                }

    # Статистика
    print(f"\n{'='*50}")
    print("СТАТИСТИКА ДАТАСЕТУ")
    print(f"{'='*50}")

    type_counts = defaultdict(int)
    for key, val in dataset.items():
        if val["type"]:
            type_counts[val["type"]] += 1

    for plag_type, count in type_counts.items():
        print(f"  {plag_type}: {count} пар")

    print(f"\n  ВСЬОГО: {len(dataset)} пар")

    return dict(dataset)


# ==============================================================================
# 4. PYTORCH DATASET
# ==============================================================================

class TripletMidiDataset(Dataset):
    """Dataset для Triplet Loss"""

    def __init__(self, dataset: Dict[str, Dict[str, np.ndarray]],
                 mode: str = "train", train_ratio: float = 0.8):
        self.dataset = dataset
        self.keys = list(dataset.keys())

        # Перемішуємо
        random.seed(42)
        random.shuffle(self.keys)

        # Розділяємо
        split_idx = int(len(self.keys) * train_ratio)
        if mode == "train":
            self.keys = self.keys[:split_idx]
        else:
            self.keys = self.keys[split_idx:]

        self.all_keys = list(dataset.keys())
        print(f"  {mode}: {len(self.keys)} зразків")

    def __len__(self) -> int:
        return len(self.keys)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        key = self.keys[idx]

        # Anchor - оригінал
        anchor = self.dataset[key]["original"]

        # Positive - плагіат того ж кейсу
        positive = self.dataset[key]["plagiarism"]

        # Negative - інший кейс
        negative_key = random.choice([k for k in self.all_keys if k != key])
        if random.random() > 0.5:
            negative = self.dataset[negative_key]["original"]
        else:
            negative = self.dataset[negative_key]["plagiarism"]

        return (
            torch.FloatTensor(anchor),
            torch.FloatTensor(positive),
            torch.FloatTensor(negative)
        )


# ==============================================================================
# 5. LSTM МОДЕЛЬ
# ==============================================================================

class LSTMEmbedding(nn.Module):
    """LSTM з Attention для музичних ембедінгів"""

    def __init__(self, input_size: int = 64,
                 hidden_size: int = 128,
                 num_layers: int = 2,
                 embedding_size: int = 64,
                 dropout: float = 0.3,
                 bidirectional: bool = True):
        super().__init__()

        self.num_directions = 2 if bidirectional else 1

        # LSTM
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional
        )

        # Attention
        att_size = hidden_size * self.num_directions
        self.attention = nn.Sequential(
            nn.Linear(att_size, att_size),
            nn.Tanh(),
            nn.Linear(att_size, 1)
        )

        # FC
        self.fc = nn.Sequential(
            nn.Linear(att_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, embedding_size)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        lstm_out, _ = self.lstm(x)

        # Attention
        att_weights = torch.softmax(self.attention(lstm_out), dim=1)
        context = torch.sum(att_weights * lstm_out, dim=1)

        # Embedding + L2 norm
        emb = self.fc(context)
        return nn.functional.normalize(emb, p=2, dim=1)


# ==============================================================================
# 6. TRIPLET LOSS
# ==============================================================================

class TripletLoss(nn.Module):
    def __init__(self, margin: float = 1.0):
        super().__init__()
        self.margin = margin

    def forward(self, anchor, positive, negative):
        dist_pos = torch.sum((anchor - positive) ** 2, dim=1)
        dist_neg = torch.sum((anchor - negative) ** 2, dim=1)
        loss = torch.relu(dist_pos - dist_neg + self.margin)
        return loss.mean()


# ==============================================================================
# 7. ТРЕНУВАННЯ
# ==============================================================================

class Trainer:
    def __init__(self, model: nn.Module, config=Config):
        self.model = model.to(config.DEVICE)
        self.config = config
        self.device = config.DEVICE

        self.criterion = TripletLoss(margin=config.MARGIN)
        self.optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE)
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', patience=5, factor=0.5
        )

        self.train_losses = []
        self.val_losses = []
        self.best_val_loss = float('inf')
        self.patience_counter = 0

    def train_epoch(self, dataloader):
        self.model.train()
        total_loss = 0.0

        for batch in tqdm(dataloader, desc="Train", leave=False):
            anchor, positive, negative = [x.to(self.device) for x in batch]

            loss = self.criterion(
                self.model(anchor),
                self.model(positive),
                self.model(negative)
            )

            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.optimizer.step()

            total_loss += loss.item()

        return total_loss / len(dataloader)

    def validate(self, dataloader):
        self.model.eval()
        total_loss = 0.0

        with torch.no_grad():
            for batch in tqdm(dataloader, desc="Val", leave=False):
                anchor, positive, negative = [x.to(self.device) for x in batch]

                loss = self.criterion(
                    self.model(anchor),
                    self.model(positive),
                    self.model(negative)
                )
                total_loss += loss.item()

        return total_loss / len(dataloader)

    def train(self, train_loader, val_loader):
        print(f"\n{'='*50}")
        print(f"🚀 Тренування на {self.device}")
        print(f"{'='*50}")

        for epoch in range(self.config.NUM_EPOCHS):
            train_loss = self.train_epoch(train_loader)
            val_loss = self.validate(val_loader)

            self.train_losses.append(train_loss)
            self.val_losses.append(val_loss)
            self.scheduler.step(val_loss)

            lr = self.optimizer.param_groups[0]['lr']
            print(f"Epoch {epoch+1:3d}/{self.config.NUM_EPOCHS} | "
                  f"Train: {train_loss:.4f} | Val: {val_loss:.4f} | LR: {lr:.6f}")

            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.patience_counter = 0
                self.save_model("best_model.pth")
                print("  ✓ Saved best model!")
            else:
                self.patience_counter += 1

            if self.patience_counter >= self.config.PATIENCE:
                print(f"\n⏹️ Early stopping at epoch {epoch + 1}")
                break

        self.save_model("final_model.pth")
        self.plot_training()

    def save_model(self, filename):
        os.makedirs(self.config.MODEL_SAVE_PATH, exist_ok=True)
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'train_losses': self.train_losses,
            'val_losses': self.val_losses,
            'config': {
                'input_size': self.config.INPUT_SIZE,
                'hidden_size': self.config.HIDDEN_SIZE,
                'num_layers': self.config.NUM_LAYERS,
                'embedding_size': self.config.EMBEDDING_SIZE,
            }
        }, os.path.join(self.config.MODEL_SAVE_PATH, filename))

    def plot_training(self):
        plt.figure(figsize=(10, 5))
        plt.plot(self.train_losses, label='Train', color='blue')
        plt.plot(self.val_losses, label='Val', color='orange')
        plt.xlabel('Epoch')
        plt.ylabel('Triplet Loss')
        plt.title('Training Progress')
        plt.legend()
        plt.grid(True)
        plt.savefig(os.path.join(self.config.MODEL_SAVE_PATH, 'training.png'), dpi=150)
        plt.close()


# ==============================================================================
# 8. ДЕТЕКТОР ПЛАГІАТУ
# ==============================================================================

class PlagiarismDetector:
    def __init__(self, model, parser, device=Config.DEVICE):
        self.model = model.to(device)
        self.model.eval()
        self.parser = parser
        self.device = device

    def get_embedding(self, midi_path: str) -> Optional[np.ndarray]:
        features = self.parser.parse_midi(midi_path)
        if features is None:
            return None

        with torch.no_grad():
            x = torch.FloatTensor(features).unsqueeze(0).to(self.device)
            return self.model(x).cpu().numpy().squeeze()

    def get_embedding_from_array(self, features: np.ndarray) -> np.ndarray:
        with torch.no_grad():
            x = torch.FloatTensor(features).unsqueeze(0).to(self.device)
            return self.model(x).cpu().numpy().squeeze()

    def distance(self, emb1, emb2) -> float:
        return float(np.linalg.norm(emb1 - emb2))

    def similarity(self, emb1, emb2) -> float:
        return float(np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2)))

    def check_plagiarism(self, midi1: str, midi2: str, threshold: float = 0.5) -> Dict:
        emb1 = self.get_embedding(midi1)
        emb2 = self.get_embedding(midi2)

        if emb1 is None or emb2 is None:
            return {"error": "Failed to parse MIDI"}

        dist = self.distance(emb1, emb2)
        sim = self.similarity(emb1, emb2)

        return {
            "is_plagiarism": dist < threshold,
            "distance": dist,
            "similarity": sim,
            "threshold": threshold
        }

    def evaluate(self, dataset: Dict) -> Dict:
        """Оцінка на датасеті"""
        print("\n📊 Оцінка моделі...")

        keys = list(dataset.keys())
        embeddings = {}

        for key in tqdm(keys, desc="Computing embeddings"):
            embeddings[key] = {
                "ori": self.get_embedding_from_array(dataset[key]["original"]),
                "plag": self.get_embedding_from_array(dataset[key]["plagiarism"])
            }

        pos_dists = []
        neg_dists = []
        correct = 0
        total = 0

        # Результати по типах плагіату
        type_results = defaultdict(lambda: {"correct": 0, "total": 0, "pos_dists": []})

        for i, key in enumerate(keys):
            plag_type = dataset[key].get("type", "unknown")

            pos_dist = self.distance(embeddings[key]["ori"], embeddings[key]["plag"])
            pos_dists.append(pos_dist)
            type_results[plag_type]["pos_dists"].append(pos_dist)

            for j, other_key in enumerate(keys):
                if i != j:
                    neg_dist = self.distance(embeddings[key]["ori"], embeddings[other_key]["ori"])
                    neg_dists.append(neg_dist)

                    if pos_dist < neg_dist:
                        correct += 1
                        type_results[plag_type]["correct"] += 1

                    total += 1
                    type_results[plag_type]["total"] += 1

        # Виводимо результати
        print(f"\n{'='*60}")
        print("РЕЗУЛЬТАТИ")
        print(f"{'='*60}")
        print(f"Overall Accuracy: {correct/total:.2%}")
        print(f"Positive Distance: {np.mean(pos_dists):.4f} ± {np.std(pos_dists):.4f}")
        print(f"Negative Distance: {np.mean(neg_dists):.4f} ± {np.std(neg_dists):.4f}")
        print(f"Separation: {np.mean(neg_dists) - np.mean(pos_dists):.4f}")

        print(f"\nПо типах плагіату:")
        for plag_type, res in type_results.items():
            acc = res["correct"] / res["total"] if res["total"] > 0 else 0
            avg_dist = np.mean(res["pos_dists"]) if res["pos_dists"] else 0
            print(f"  {plag_type:10s}: Acc={acc:.2%}, Avg Pos Dist={avg_dist:.4f}")

        return {
            "accuracy": correct / total,
            "pos_dist_mean": np.mean(pos_dists),
            "neg_dist_mean": np.mean(neg_dists),
            "type_results": dict(type_results)
        }


# ==============================================================================
# 9. СИНТЕТИЧНІ ДАНІ
# ==============================================================================

def create_synthetic_dataset(num_cases: int = 200) -> Dict:
    """Синтетичний датасет для тестування"""
    print(f"\n🎵 Створення синтетичних даних ({num_cases} пар)...")

    dataset = {}
    types = ["duration", "melody", "pitch", "shuffle"]

    for i in range(num_cases):
        plag_type = types[i % 4]
        key = f"{plag_type}_Case{i:04d}"

        # Base pattern
        base = np.zeros((Config.MAX_SEQ_LENGTH, 128))
        melody = np.random.randint(40, 80, size=Config.MAX_SEQ_LENGTH // 10)
        for j, note in enumerate(melody):
            start = j * 10
            base[start:min(start+8, Config.MAX_SEQ_LENGTH), note] = np.random.uniform(0.5, 1.0)

        original = base + np.random.randn(*base.shape) * 0.05
        original = np.clip(original, 0, 1).astype(np.float32)

        # Plagiarism з різними трансформаціями
        plagiarism = base.copy()
        if plag_type == "pitch":
            plagiarism = np.roll(plagiarism, np.random.randint(-3, 4), axis=1)
        elif plag_type == "duration":
            # Stretch/compress
            factor = np.random.uniform(0.9, 1.1)
            indices = np.linspace(0, len(plagiarism)-1, int(len(plagiarism)*factor)).astype(int)
            indices = np.clip(indices[:Config.MAX_SEQ_LENGTH], 0, len(plagiarism)-1)
            plagiarism = plagiarism[indices]
            if len(plagiarism) < Config.MAX_SEQ_LENGTH:
                pad = np.zeros((Config.MAX_SEQ_LENGTH - len(plagiarism), 128))
                plagiarism = np.vstack([plagiarism, pad])
        elif plag_type == "shuffle":
            # Переставляємо частини
            chunks = np.array_split(plagiarism, 4)
            np.random.shuffle(chunks)
            plagiarism = np.vstack(chunks)[:Config.MAX_SEQ_LENGTH]

        plagiarism = plagiarism + np.random.randn(*plagiarism.shape) * 0.05
        plagiarism = np.clip(plagiarism, 0, 1).astype(np.float32)

        dataset[key] = {
            "original": original,
            "plagiarism": plagiarism,
            "type": plag_type
        }

    print(f"✓ Створено {len(dataset)} пар")
    return dataset


# ==============================================================================
# 10. ГОЛОВНА ФУНКЦІЯ
# ==============================================================================

def main():
    print("="*60)
    print("🎵 LSTM + Triplet Loss - Music Plagiarism Detection")
    print("="*60)

    parser = MidiParser(
        max_length=Config.MAX_SEQ_LENGTH,
        sample_rate=Config.SAMPLE_RATE
    )

    # Перевіряємо наявність даних
    has_data = os.path.exists(Config.ORI_DIR) and os.path.exists(Config.PLAG_DIR)

    if has_data:
        print(f"\n✓ Знайдено датасет")
        print(f"  Оригінали: {Config.ORI_DIR}")
        print(f"  Плагіати: {Config.PLAG_DIR}")

        dataset = load_midi_dataset_with_subfolders(
            Config.ORI_DIR,
            Config.PLAG_DIR,
            parser,
            Config.PLAGIARISM_TYPES
        )

        if len(dataset) < 10:
            print("⚠️ Замало даних, використовую синтетичні")
            dataset = create_synthetic_dataset(200)
    else:
        print(f"\n⚠️ Датасет не знайдено")
        print(f"   Очікується структура:")
        print(f"   {Config.ORI_DIR}/")
        print(f"   ├── duration/")
        print(f"   ├── melody/")
        print(f"   ├── pitch/")
        print(f"   └── shuffle/")
        print(f"\nВикористовую синтетичні дані...")
        dataset = create_synthetic_dataset(200)

    # Визначаємо розмір входу
    sample = list(dataset.values())[0]["original"]
    Config.INPUT_SIZE = sample.shape[1]
    print(f"\nInput size: {Config.INPUT_SIZE}, Seq length: {sample.shape[0]}")

    # DataLoaders
    train_ds = TripletMidiDataset(dataset, mode="train")
    val_ds = TripletMidiDataset(dataset, mode="val")

    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE,
                              shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE,
                            shuffle=False, drop_last=True)

    # Модель
    model = LSTMEmbedding(
        input_size=Config.INPUT_SIZE,
        hidden_size=Config.HIDDEN_SIZE,
        num_layers=Config.NUM_LAYERS,
        embedding_size=Config.EMBEDDING_SIZE,
        dropout=Config.DROPOUT,
        bidirectional=Config.BIDIRECTIONAL
    )

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {total_params:,}")

    # Тренування
    trainer = Trainer(model, Config)
    trainer.train(train_loader, val_loader)

    # Оцінка
    detector = PlagiarismDetector(model, parser, Config.DEVICE)
    results = detector.evaluate(dataset)

    print(f"\n{'='*60}")
    print("✓ DONE!")
    print(f"{'='*60}")
    print(f"Models saved to: {Config.MODEL_SAVE_PATH}/")

    return model, detector, results


model, detector, results = main()

🎵 LSTM + Triplet Loss - Music Plagiarism Detection

✓ Знайдено датасет
  Оригінали: ./data/midi/MPD-Set_ori
  Плагіати: ./data/midi/MPD-Set_plag

📁 Завантаження типу: duration
   Оригіналів: 250, Плагіатів: 250, Пар: 0



📁 Завантаження типу: melody
   Оригіналів: 250, Плагіатів: 250, Пар: 0



📁 Завантаження типу: pitch
   Оригіналів: 250, Плагіатів: 250, Пар: 0



📁 Завантаження типу: shuffle
   Оригіналів: 250, Плагіатів: 250, Пар: 250



СТАТИСТИКА ДАТАСЕТУ
  shuffle: 250 пар

  ВСЬОГО: 250 пар

Input size: 128, Seq length: 500
  train: 200 зразків
  val: 50 зразків
Model parameters: 227,713

🚀 Тренування на cpu


...

Epoch  18/20 | Train: 0.7928 | Val: 0.9884 | LR: 0.000500


Epoch  19/20 | Train: 0.7791 | Val: 1.0847 | LR: 0.000500


Epoch  20/20 | Train: 0.7536 | Val: 1.1089 | LR: 0.000500

📊 Оцінка моделі...


Computing embeddings: 100%|██████████| 250/250 [00:07<00:00, 33.35it/s]



РЕЗУЛЬТАТИ
Overall Accuracy: 72.33%
Positive Distance: 0.3122 ± 0.5108
Negative Distance: 0.6168 ± 0.5929
Separation: 0.3046

По типах плагіату:
  shuffle   : Acc=72.33%, Avg Pos Dist=0.3122

✓ DONE!
Models saved to: ./models/
